# LSTM — prevendo o gasto do próximo mês (Módulo 9)

Volta pro dado real do Projeto Lupa: usar o histórico mensal de gasto de cada deputado pra prever o gasto do mês seguinte.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from src.preprocessing import get_engine

torch.manual_seed(42)

engine = get_engine()
query = """
    SELECT f.deputado_id, t.ano, t.mes, SUM(f.valor_documento) AS gasto_mes
    FROM fact_despesas f
    JOIN dim_tempo t ON t.id_tempo = f.id_tempo
    GROUP BY f.deputado_id, t.ano, t.mes
    ORDER BY f.deputado_id, t.ano, t.mes
"""
gasto_mensal = pd.read_sql(query, engine)
gasto_mensal["mes_ref"] = pd.to_datetime(dict(year=gasto_mensal["ano"], month=gasto_mensal["mes"], day=1))
print(gasto_mensal.shape)
gasto_mensal.head()

(19591, 5)


,deputado_id,ano,mes,gasto_mes,mes_ref
0,62881,2023,1,8205.52,2023-01-01
1,62881,2023,2,35296.83,2023-02-01
2,62881,2023,3,46124.23,2023-03-01
3,62881,2023,4,40364.97,2023-04-01
4,62881,2023,5,39044.25,2023-05-01


## Preenchendo lacunas e montando as sequências

Nem todo deputado tem despesa lançada todo mês (podem ter meses de gasto zero). Preenchemos esses buracos com 0 pra manter a série cronológica contínua, senão a LSTM aprenderia uma sequência de tempo errada.

In [2]:
todos_meses = pd.date_range(gasto_mensal["mes_ref"].min(), gasto_mensal["mes_ref"].max(), freq="MS")
deputados_validos = gasto_mensal["deputado_id"].unique()

pivot = gasto_mensal.pivot_table(index="deputado_id", columns="mes_ref", values="gasto_mes", fill_value=0)
pivot = pivot.reindex(columns=todos_meses, fill_value=0)

# só deputados com histórico longo o suficiente pra formar pelo menos 1 janela + alvo
TAMANHO_JANELA = 6
deputados_com_historico = pivot.index[(pivot != 0).sum(axis=1) >= TAMANHO_JANELA + 3]
pivot = pivot.loc[deputados_com_historico]
print(f"Deputados usados: {len(pivot)} | Meses no histórico: {pivot.shape[1]}")

# escala em milhares, pra facilitar o treino da rede
valores = pivot.values / 1000.0

Deputados usados: 487 | Meses no histórico: 45


In [3]:
n_meses = valores.shape[1]
CORTE_TESTE = n_meses - 6  # últimos 6 meses viram alvo de teste - out-of-time, mesmo princípio dos outros módulos

X_treino_seq, y_treino_seq, X_teste_seq, y_teste_seq = [], [], [], []

for serie in valores:
    for fim_janela in range(TAMANHO_JANELA, n_meses):
        janela = serie[fim_janela - TAMANHO_JANELA : fim_janela]
        alvo = serie[fim_janela]
        if fim_janela < CORTE_TESTE:
            X_treino_seq.append(janela)
            y_treino_seq.append(alvo)
        else:
            X_teste_seq.append(janela)
            y_teste_seq.append(alvo)

X_treino_seq = torch.tensor(np.array(X_treino_seq), dtype=torch.float32).unsqueeze(-1)
y_treino_seq = torch.tensor(np.array(y_treino_seq), dtype=torch.float32).unsqueeze(-1)
X_teste_seq = torch.tensor(np.array(X_teste_seq), dtype=torch.float32).unsqueeze(-1)
y_teste_seq = torch.tensor(np.array(y_teste_seq), dtype=torch.float32).unsqueeze(-1)

print(f"Treino: {X_treino_seq.shape} | Teste: {X_teste_seq.shape}")

Treino: torch.Size([16071, 6, 1]) | Teste: torch.Size([2922, 6, 1])


## LSTM vs. baseline ingênuo (repetir o último mês)

Um modelo só é útil se bater um baseline simples. Aqui, o baseline "ingênuo" é prever que o próximo mês será igual ao último mês observado — surpreendentemente forte em séries com pouca variação brusca.

In [4]:
class LSTMGasto(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.saida = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.saida(h_n[-1])


modelo_lstm = LSTMGasto()
otimizador_lstm = torch.optim.Adam(modelo_lstm.parameters(), lr=1e-3)
perda_lstm = nn.MSELoss()

TAMANHO_LOTE_LSTM = 256
N_EPOCAS_LSTM = 50
n_amostras_lstm = X_treino_seq.shape[0]

for epoca in range(N_EPOCAS_LSTM):
    modelo_lstm.train()
    permutacao = torch.randperm(n_amostras_lstm)
    perda_epoca = 0.0
    for i in range(0, n_amostras_lstm, TAMANHO_LOTE_LSTM):
        idx_lote = permutacao[i : i + TAMANHO_LOTE_LSTM]
        otimizador_lstm.zero_grad()
        saida = modelo_lstm(X_treino_seq[idx_lote])
        perda = perda_lstm(saida, y_treino_seq[idx_lote])
        perda.backward()
        otimizador_lstm.step()
        perda_epoca += perda.item() * len(idx_lote)
    if (epoca + 1) % 10 == 0:
        print(f"Época {epoca+1}/{N_EPOCAS_LSTM} - perda média (MSE, escala mil R$): {perda_epoca/n_amostras_lstm:.4f}")

modelo_lstm.eval()
with torch.no_grad():
    pred_lstm = modelo_lstm(X_teste_seq)

mae_lstm = (pred_lstm - y_teste_seq).abs().mean().item() * 1000  # volta pra R$
mae_baseline = (X_teste_seq[:, -1, :] - y_teste_seq).abs().mean().item() * 1000

print(f"\nMAE LSTM: R$ {mae_lstm:,.2f}")
print(f"MAE baseline (repete último mês): R$ {mae_baseline:,.2f}")

Época 10/50 - perda média (MSE, escala mil R$): 709.2698


Época 20/50 - perda média (MSE, escala mil R$): 370.0765


Época 30/50 - perda média (MSE, escala mil R$): 299.0451


Época 40/50 - perda média (MSE, escala mil R$): 285.8999


Época 50/50 - perda média (MSE, escala mil R$): 283.0212

MAE LSTM: R$ 17,186.58
MAE baseline (repete último mês): R$ 16,778.56
